	Significant Digits



In [ ]:
from decimal import Decimal, ROUND_HALF_EVEN
#decimal & banker
value = input("Enter value: ")
n = int(input("Enter number of significant figures: "))

x = Decimal(value)

# Find the decimal position needed
shift = n - x.adjusted() - 1

quantize_value = Decimal("1e" + str(-shift))

result = x.quantize(quantize_value, rounding=ROUND_HALF_EVEN)

print("Result:", result)

Enter value:  30.0567
Enter number of significant figures:  4


Result: 30.06


QUestion jevabe aste pare  

In [15]:
def bankers_round_sig_fig(num_str, sig_figs):
    """
    Rounds a number string to 'sig_figs' significant figures using Banker's Rounding
    without using decimal library or built-in round().
    """
    num_str = num_str.strip()
    sign = ""
    if num_str.startswith("-"):
        sign = "-"
        num_str = num_str[1:]

    # Separate integer and fractional parts
    if "." in num_str:
        int_part, frac_part = num_str.split(".")
    else:
        int_part, frac_part = num_str, ""

    full_str = int_part + frac_part

    # 1. Find the first non-zero digit (1st significant figure)
    first_nz = -1
    for idx, c in enumerate(full_str):
        if c != '0':
            first_nz = idx
            break

    # If number is zero
    if first_nz == -1:
        return sign + "0"

    # 2. Determine position of 1st significant figure relative to decimal point
    if first_nz < len(int_part):
        first_power = len(int_part) - 1 - first_nz
    else:
        first_power = -(first_nz - len(int_part) + 1)

    target_power = first_power - (sig_figs - 1)
    target_idx = first_nz + sig_figs

    # Pad with extra zeros for safe checking of dropped digits
    full_str = full_str.ljust(target_idx + 10, '0')

    # 3. Analyze dropped digits for Banker's Rounding
    dropped_first = int(full_str[target_idx])
    rest_dropped = full_str[target_idx + 1:]

    if dropped_first > 5:
        go_up = True
    elif dropped_first < 5:
        go_up = False
    else:  # dropped_first == 5
        if any(c != '0' for c in rest_dropped):
            go_up = True
        else:
            # Exactly 0.5: round to nearest even (check last kept digit)
            last_kept = int(full_str[target_idx - 1])
            go_up = (last_kept % 2 != 0)

    # 4. Increment kept digits if rounding up
    kept_digits = full_str[:target_idx]
    val = int(kept_digits)
    if go_up:
        val += 1

    val_str = str(val)

    # 5. Format output string back to standard notation
    if target_power < 0:
        d = -target_power
        val_str = val_str.zfill(d + 1)
        res_int = val_str[:-d]
        res_frac = val_str[-d:]
        result = f"{res_int}.{res_frac}"
    else:
        result = val_str + ("0" * target_power)

    return sign + result


# --- Exam Table Input / Output ---

count = int(input("How many numbers? "))
sig_figs = int(input("How many significant figures? "))

numbers = []
for i in range(1, count + 1):
    numbers.append(input(f"Enter number {i}: "))

# Print Output Table
col2 = "Significant Digits"
print(f"\n{'Number':<16}{col2}")
print("-" * 35)

for num_str in numbers:
    rounded = bankers_round_sig_fig(num_str, sig_figs)
    print(f"{num_str:<16}{rounded}")

How many numbers? 4
How many significant figures? 3
Enter number 1: 2.345600
Enter number 2: 2.344500
Enter number 3: 0.0012355
Enter number 4: 1234.56

Number          Significant Digits
-----------------------------------
2.345600        2.35
2.344500        2.34
0.0012355       0.00124
1234.56         1230


In [ ]:
value = float(input("Enter value: "))
n = int(input("Enter number of significant figures: "))

# Find the number of digits before the decimal point
temp = abs(value)
digits = 0

while temp >= 1:
    temp = temp / 10
    digits = digits + 1

# Find how many decimal places to keep
decimal_places = n - digits

# Create rounding factor
factor = 1

if decimal_places > 0:
    i = 0
    while i < decimal_places:
        factor = factor * 10
        i = i + 1

    result = int(value * factor + 0.5) / factor

elif decimal_places == 0:
    result = int(value + 0.5)

else:
    i = 0
    while i < -decimal_places:
        factor = factor * 10
        i = i + 1

    result = int(value / factor + 0.5) * factor

print("Result:", result)


Enter value:  30.0567
Enter number of significant figures:  4


Result: 30.06


#### Banker’s Rounding Rule

In [6]:
def bankers_round(num_str, d):
    """
    Rounds a number string to 'd' decimal places using Banker's Rounding
    without using any external libraries or the built-in round() function.
    """
    num_str = num_str.strip()
    sign = ""
    if num_str.startswith("-"):
        sign = "-"
        num_str = num_str[1:]

    # Separate integer and fractional parts
    if "." in num_str:
        integer_part, frac_part = num_str.split(".")
    else:
        integer_part, frac_part = num_str, ""

    # Ensure fractional part is long enough
    frac_part = frac_part.ljust(d + 10, '0')

    # Dropped digit right after target decimal place 'd'
    first_dropped = int(frac_part[d])
    rest_dropped = frac_part[d + 1:]

    # Determine whether to round up or down
    if first_dropped > 5:
        go_up = True
    elif first_dropped < 5:
        go_up = False
    else:  # first_dropped == 5
        # Check if there are non-zero digits after 5
        if any(c != '0' for c in rest_dropped):
            go_up = True
        else:
            # Exactly 0.5: check last kept digit
            if d > 0:
                last_kept = int(frac_part[d - 1])
            else:
                last_kept = int(integer_part[-1])

            # If last kept digit is odd -> round up (make even)
            # If last kept digit is even -> stay
            go_up = (last_kept % 2 != 0)

    # Perform increment on the kept digits
    kept_str = integer_part + (frac_part[:d] if d > 0 else "")
    val = int(kept_str)
    if go_up:
        val += 1

    val_str = str(val)

    # Reconstruct integer and fractional parts
    if d > 0:
        val_str = val_str.zfill(len(integer_part) + d)
        res_int = val_str[:-d] if len(val_str) > d else "0"
        res_frac = val_str[-d:]
        result = f"{res_int}.{res_frac}"
    else:
        result = val_str

    return sign + result


# --- Main Input / Output Program ---

count = int(input("How many numbers? "))
d = int(input("How many decimal places? "))

numbers = []
for i in range(1, count + 1):
    val_input = input(f"Enter number {i}: ")
    numbers.append(val_input)

# Print Table Output
col2 = "Banker's Rounding"
print(f"\n{'Number':<14}{col2}")
print("-" * 31)

for num_str in numbers:
    rounded = bankers_round(num_str, d)
    print(f"{num_str:<14}{rounded}")

How many numbers? 4
How many decimal places? 3
Enter number 1: 2.345600
Enter number 2: 2.344500
Enter number 3: 2.345500
Enter number 4: 7.126500

Number        Banker's Rounding
-------------------------------
2.345600      2.346
2.344500      2.344
2.345500      2.346
7.126500      7.126


#### Horner’s Method

In [2]:
def parse_number(s):
    """Converts input string to integer if whole number, else float."""
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Inputs matching exact prompt format
n = int(input("Enter degree: "))
coeff = [parse_number(c) for c in input("Enter coefficients (highest power first): ").split()]
x = parse_number(input("Enter value of x: "))

# 2. Table Header
print("Power   a(k)       P(k)")
print("-" * 24)

# 3. Initial Horner step (Power = n)
power = n
P = coeff[0]
print(f"{power:<8}{coeff[0]:<11}{P}")

# 4. Iterative Horner steps
for i in range(1, n + 1):
    power = n - i
    P = P * x + coeff[i]
    print(f"{power:<8}{coeff[i]:<11}{P}")

# 5. Final output format
print()
print(f"f({x}) = {P}")

Enter degree: 4
Enter coefficients (highest power first): 2 -3 0 5 -7
Enter value of x: 2
Power   a(k)       P(k)
------------------------
4       2          2
3       -3         1
2       0          2
1       5          9
0       -7         11

f(2) = 11


#### Bisection Method


In [8]:
# Define the function f(x) = x^3 - x - 2
def f(x):
    return x**3 - x - 2

# Input values
a = float(input("Enter lower limit a: "))
b = float(input("Enter upper limit b: "))
tol = float(input("Enter tolerance: "))

# Check validity of initial interval
if f(a) * f(b) >= 0:
    print("Invalid interval: f(a) and f(b) must have opposite signs.")
else:
    # Print Table Header (matching Sir's slide format)
    print(f"\n{'Iteration':<13}{'a':<11}{'b':<11}{'c':<11}{'f(c)'}")
    print("-" * 55)

    iteration = 1
    c = a

    # Bisection Method Loop
    while (b - a) / 2 > tol:
        c = (a + b) / 2
        fc = f(c)

        # Print table row matching slide formatting
        print(f"{iteration:<13}{a:<11.5g}{b:<11.5g}{c:<11.5g}{fc:.4f}")

        if fc == 0:
            break
        elif f(a) * fc < 0:
            b = c
        else:
            a = c

        iteration += 1

    # Print final root
    print("-" * 55)
    print(f"Root = {c}")

Enter lower limit a: 1
Enter upper limit b: 2
Enter tolerance: 0.01

Iteration    a          b          c          f(c)
-------------------------------------------------------
1            1          2          1.5        -0.1250
2            1.5        2          1.75       1.6094
3            1.5        1.75       1.625      0.6660
4            1.5        1.625      1.5625     0.2522
5            1.5        1.5625     1.5312     0.0591
6            1.5        1.5312     1.5156     -0.0341
-------------------------------------------------------
Root = 1.515625


**SIR JODI QUESTION ae formate a kore **

In [9]:
# Evaluate polynomial using Horner's Rule
def f(coeffs, x):
    result = coeffs[0]
    for c in coeffs[1:]:
        result = result * x + c
    return result

# 1. Dynamic Inputs
n = int(input("Enter degree: "))
coeffs = [float(c) for c in input("Enter coefficients (highest power first): ").split()]
a = float(input("Enter lower limit a: "))
b = float(input("Enter upper limit b: "))
tol = float(input("Enter tolerance: "))

# 2. Check Validity of Interval
if f(coeffs, a) * f(coeffs, b) >= 0:
    print("Invalid interval: f(a) and f(b) must have opposite signs.")
else:
    # 3. Print Table Header
    print(f"\n{'Iteration':<13}{'a':<11}{'b':<11}{'c':<11}{'f(c)'}")
    print("-" * 55)

    iteration = 1
    c = a

    # 4. Bisection Loop
    while (b - a) / 2 > tol:
        c = (a + b) / 2
        fc = f(coeffs, c)

        # Print iteration row
        print(f"{iteration:<13}{a:<11.5g}{b:<11.5g}{c:<11.5g}{fc:.4f}")

        if fc == 0:
            break
        elif f(coeffs, a) * fc < 0:
            b = c
        else:
            a = c

        iteration += 1

    print("-" * 55)
    print(f"Root = {c}")

Enter degree: 3
Enter coefficients (highest power first): 1 0 -1 -2
Enter lower limit a: 1
Enter upper limit b: 2
Enter tolerance: 0.001

Iteration    a          b          c          f(c)
-------------------------------------------------------
1            1          2          1.5        -0.1250
2            1.5        2          1.75       1.6094
3            1.5        1.75       1.625      0.6660
4            1.5        1.625      1.5625     0.2522
5            1.5        1.5625     1.5312     0.0591
6            1.5        1.5312     1.5156     -0.0341
7            1.5156     1.5312     1.5234     0.0123
8            1.5156     1.5234     1.5195     -0.0110
9            1.5195     1.5234     1.5215     0.0006
-------------------------------------------------------
Root = 1.521484375


In [ ]:
def f(x):
    return x**5 - x - 1

    #x**3 - x - 2

a = float(input("Enter x1: "))
b = float(input("Enter x2: "))
E = float(input("Enter error E: "))

f1 = f(a)
f2 = f(b)

if f1 * f2 > 0:
    print("No root in this interval")
else:
    while abs(b - a) >= E:
        c = (a + b) / 2
        f0 = f(c)

        if f1 * f0 < 0:
            b = c
            f2 = f0
        else:
            a = c
            f1 = f0

    root = (a + b) / 2
    print("Root =", root)
    print("f(root) =", f(root))

Enter x1:  1
Enter x2:  2
Enter error E:  0.001


Root = 1.16748046875
f(root) = 0.0014624323171985587


# False Position Method

In [ ]:
def f(x):
    return x**2 - x - 2

x1 = float(input("Enter x1: "))
x2 = float(input("Enter x2: "))
E = float(input("Enter error E: "))

f1 = f(x1)
f2 = f(x2)

if f1 * f2 >= 0:
    print("Invalid interval: no sign change")
else:
    x0_old = None

    while True:
        x0 = x1 - (f1 * (x2 - x1)) / (f2 - f1)
        f0 = f(x0)

        if x0_old is not None and abs(x0 - x0_old) < E:
            break

        if f1 * f0 < 0:
            x2 = x0
            f2 = f0
        else:
            x1 = x0
            f1 = f0

        x0_old = x0

    print("Root =", x0)
    print("f(root) =", f(x0))

Enter x1:  1
Enter x2:  3
Enter error E:  0.001


Root = 1.9999084500595075
f(root) = -0.0002746414400860431


**SIR JODI QUESTION ae formate a kore **

In [ ]:
False Position Method

In [10]:
# Evaluate polynomial dynamically using Horner's Rule
def f(coeffs, x):
    result = coeffs[0]
    for c in coeffs[1:]:
        result = result * x + c
    return result

# 1. Dynamic Inputs
n = int(input("Enter degree: "))
coeffs = [float(c) for c in input("Enter coefficients (highest power first): ").split()]
a = float(input("Enter lower limit a: "))
b = float(input("Enter upper limit b: "))
E = float(input("Enter error tolerance E: "))

# 2. Check Validity of Initial Interval
fa = f(coeffs, a)
fb = f(coeffs, b)

if fa * fb >= 0:
    print("Invalid interval: f(a) and f(b) must have opposite signs.")
else:
    # 3. Print Slide Table Header
    print(f"\n{'Iteration':<11}{'a':<10}{'b':<10}{'f(a)':<11}{'f(b)':<11}{'x_n':<10}{'f(x_n)':<11}{'Updated Interval'}")
    print("-" * 88)

    iteration = 1
    x_prev = None

    # 4. False Position Iteration Loop
    while True:
        fa = f(coeffs, a)
        fb = f(coeffs, b)

        # Regula Falsi formula
        xn = (a * fb - b * fa) / (fb - fa)
        fxn = f(coeffs, xn)

        # Check stopping condition (converged up to tolerance / decimal match)
        is_converged = (x_prev is not None and abs(xn - x_prev) < E)

        if is_converged:
            updated_interval = "Converged"
            print(f"{iteration:<11}{a:<10.4f}{b:<10.4f}{fa:<11.4f}{fb:<11.4f}{xn:<10.4f}{fxn:<11.4f}{updated_interval}")
            break
        else:
            # Determine new interval bounds
            if fa * fxn < 0:
                next_a, next_b = a, xn
            else:
                next_a, next_b = xn, b

            updated_interval = f"[{next_a:.4f}, {next_b:.4f}]"
            print(f"{iteration:<11}{a:<10.4f}{b:<10.4f}{fa:<11.4f}{fb:<11.4f}{xn:<10.4f}{fxn:<11.4f}{updated_interval}")

            a, b = next_a, next_b
            x_prev = xn
            iteration += 1

    # 5. Final Output Line (matching slide format)
    print()
    print(f"Root ≈ {xn:.4f} (after {iteration} iterations)")

Enter degree: 3
Enter coefficients (highest power first): 1 0 -2 -5
Enter lower limit a: 2
Enter upper limit b: 3
Enter error tolerance E: 0.001

Iteration  a         b         f(a)       f(b)       x_n       f(x_n)     Updated Interval
----------------------------------------------------------------------------------------
1          2.0000    3.0000    -1.0000    16.0000    2.0588    -0.3908    [2.0588, 3.0000]
2          2.0588    3.0000    -0.3908    16.0000    2.0813    -0.1472    [2.0813, 3.0000]
3          2.0813    3.0000    -0.1472    16.0000    2.0896    -0.0547    [2.0896, 3.0000]
4          2.0896    3.0000    -0.0547    16.0000    2.0927    -0.0202    [2.0927, 3.0000]
5          2.0927    3.0000    -0.0202    16.0000    2.0939    -0.0075    [2.0939, 3.0000]
6          2.0939    3.0000    -0.0075    16.0000    2.0943    -0.0027    Converged

Root ≈ 2.0943 (after 6 iterations)


In [ ]:
def f(x):
    return x**2 - x - 2

x1 = float(input("Enter x1: "))
x2 = float(input("Enter x2: "))
E = float(input("Enter error E: "))

f1 = f(x1)
f2 = f(x2)

if f1 * f2 >= 0:
    print("Invalid interval: no sign change")
else:
    x0_old = None

    while True:
        x0 = x1 - (f1 * (x2 - x1)) / (f2 - f1)
        f0 = f(x0)

        if x0_old is not None and abs(x0 - x0_old) < E:
            break

        if f1 * f0 < 0:
            x2 = x0
            f2 = f0
        else:
            x1 = x0
            f1 = f0

        x0_old = x0

    print("Root =", x0)
    print("f(root) =", f(x0))

Enter x1:  1
Enter x2:  3
Enter error E:  0.001


Root = 1.9999084500595075
f(root) = -0.0002746414400860431


# Fixed Point Iteration Method

In [ ]:
def g(x):
    return (x + 2)**(1/3)

x = float(input("Enter initial guess: "))
E = float(input("Enter error E: "))

while True:
    x_new = g(x)

    if abs(x_new - x) < E:
        break

    x = x_new

print("Root =", x_new)

Enter initial guess:  1.5
Enter error E:  0.001


Root = 1.52131569819048


If
f(a)×f(b)<0
f(a)×f(b)<0, the signs are opposite, so a root is bracketed (assuming f is continuous).

If
f(a)×f(b)>0
f(a)×f(b)>0, the interval does not bracket a root by a sign change


Yes\! ✅ You understood it correctly. Let me show you the steps for the Bisection Method.

## Step 1: Check the signs

Calculate $f(a)$ and $f(b)$.

- If $f(a)\times f(b)<0$, the signs are opposite, so a root is bracketed (assuming $f$ is continuous).
- If $f(a)\times f(b)>0$, the interval does not bracket a root by a sign change.

## Step 2: Find the middle value

If the signs are opposite, calculate:

$$
\boxed{c=\frac{a+b}{2}}
$$

For example, suppose $a=1$ and $b=2$.

$$
c=\frac{1+2}{2}=1.5
$$

## Step 3: Check the new midpoint

For $f(x)=x^3-x-2$:

- $f(1)=-2$
- $f(2)=4$
- $f(1.5)=-0.125$

Since $f(1.5)$ is negative and $f(2)$ is positive, the new interval becomes $[1.5,2]$.

Remember this flow:

Check $f(a)f(b)<0$

Find $c=(a+b)/2$

Check signs again and keep the half containing the root

Important: This is the Bisection Method. In Fixed Point Iteration, you instead start with a guess and calculate the next value using $x_{n+1}=g(x_n)$; you don't need to find a midpoint.

Newton-Raphson Method

In [ ]:
#f(x)=x^2−3x+2:
def f(x):
    return x**2 - 3*x + 2

def df(x):
    return 2*x - 3

x = float(input("Enter initial guess x0: "))
E = float(input("Enter error E: "))

while True:
    if df(x) == 0:
        print("Derivative is zero. Cannot continue.")
        break

    x_new = x - f(x) / df(x)

    if abs(x_new - x) < E:
        print("Root =", x_new)
        break

    x = x_new

Enter initial guess x0:  0
Enter error E:  0.001


Root = 0.9999999997671692


In [13]:
# Helper to convert float to int if whole number
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Horner's rule to evaluate polynomial f(x)
def f(coeffs, x):
    result = coeffs[0]
    for c in coeffs[1:]:
        result = result * x + c
    return result

# 2. Function to compute derivative coefficients automatically
def get_derivative_coeffs(coeffs):
    n = len(coeffs) - 1
    d_coeffs = []
    for i in range(n):
        power = n - i
        d_coeffs.append(coeffs[i] * power)
    return d_coeffs

# 3. Inputs matching exam style
n = int(input("Enter degree: "))
coeffs = [float(c) for c in input("Enter coefficients (highest power first): ").split()]
x0_raw = parse_val(input("Enter initial guess x0: "))
x = float(x0_raw)
E = float(input("Enter error tolerance E: "))

# Calculate derivative coefficients automatically
d_coeffs = get_derivative_coeffs(coeffs)

# 4. Table Header
headers = ["n", "x(n)", "f(x(n))", "f'(x(n))", "x(n+1)"]
print(f"\n{headers[0]:<6}{headers[1]:<11}{headers[2]:<11}{headers[3]:<11}{headers[4]}")
print("-" * 52)

step = 1

# 5. Newton-Raphson Iteration Loop
while True:
    fx = f(coeffs, x)
    dfx = f(d_coeffs, x)

    # Division by zero check
    if dfx == 0:
        print("Derivative is zero. Cannot continue.")
        break

    # Newton-Raphson formula: x(n+1) = x(n) - f(x(n)) / f'(x(n))
    x_next = x - fx / dfx

    # Print iteration row formatted to 4 decimal places
    print(f"{step:<6}{x:<11.4f}{fx:<11.4f}{dfx:<11.4f}{x_next:.4f}")

    # Check convergence condition
    if abs(x_next - x) < E:
        break

    x = x_next
    step += 1

# 6. Final Output
print()
print(f"Root = {x_next:.4f}")

Enter degree: 2
Enter coefficients (highest power first): 1 -3 2
Enter initial guess x0: 0
Enter error tolerance E: 0.0001

n     x(n)       f(x(n))    f'(x(n))   x(n+1)
----------------------------------------------------
1     0.0000     2.0000     -3.0000    0.6667
2     0.6667     0.4444     -1.6667    0.9333
3     0.9333     0.0711     -1.1333    0.9961
4     0.9961     0.0039     -1.0078    1.0000
5     1.0000     0.0000     -1.0000    1.0000

Root = 1.0000


Secant method  

In [4]:
# Horner's Rule function to evaluate polynomial f(x)
def f(coeffs, x):
    result = coeffs[0]
    for c in coeffs[1:]:
        result = result * x + c
    return result

# 1. Inputs matching exact prompt format
n = int(input("Enter degree: "))
coeffs = [float(c) for c in input("Enter coefficients (highest power first): ").split()]
x0, x1 = map(float, input("Enter x0 and x1: ").split())
decimals = int(input("Correct up to how many decimal places? "))

# 2. Check validity of initial values
f_x0 = f(coeffs, x0)
f_x1 = f(coeffs, x1)

if f_x0 * f_x1 >= 0:
    print("Invalid initial values")
else:
    # 3. Print Table Header
    print(f"{'i':<5}{'x(i)':<11}{'x(i-1)':<11}{'f(x(i))':<11}{'f(x(i-1))':<11}{'x(i+1)'}")
    print("-" * 55)

    i = 1
    curr_x = x1  # x(i)
    prev_x = x0  # x(i-1)

    while True:
        f_curr = f(coeffs, curr_x)
        f_prev = f(coeffs, prev_x)

        # Check division by zero
        if f_curr == f_prev:
            print("Division by zero")
            break

        # Secant Method formula
        next_x = curr_x - (f_curr * (curr_x - prev_x)) / (f_curr - f_prev)

        # Print iteration row with 5 decimal places for table alignment
        print(f"{i:<5}{curr_x:<11.5f}{prev_x:<11.5f}{f_curr:<11.5f}{f_prev:<11.5f}{next_x:.5f}")

        # 4. Stop when two successive values match up to given decimal places
        if f"{curr_x:.{decimals}f}" == f"{next_x:.{decimals}f}":
            print()
            print(f"Root = {next_x:.{decimals}f}")
            break

        prev_x = curr_x
        curr_x = next_x
        i += 1

Enter degree: 3
Enter coefficients (highest power first): 2 0 -2 -5
Enter x0 and x1: 1 2 
Correct up to how many decimal places? 2
i    x(i)       x(i-1)     f(x(i))    f(x(i-1))  x(i+1)
-------------------------------------------------------
1    2.00000    1.00000    7.00000    -5.00000   1.41667
2    1.41667    2.00000    -2.14699   7.00000    1.55359
3    1.55359    1.41667    -0.60759   -2.14699   1.60763
4    1.60763    1.55359    0.09449    -0.60759   1.60036
5    1.60036    1.60763    -0.00324   0.09449    1.60060

Root = 1.60


Remember these three steps Calculate f(x1)f(x 1​) and f(x2)f(x 2​).

Use the Secant formula to calculate
x3x 3​ .
If ∣x3x2∣<E∣x 3​ −x 2​ ∣<E, stop; otherwise, set
x1=x2x 1​=x 2​  and x2 =
x3x 2​ =x 3​ , then repeat.

In [ ]:
Newton’s Forward & Backward Interpolation

In [ ]:
# Newton Forward and Backward Interpolation

n = int(input("Enter number of data points: "))

x = []
y = []

for i in range(n):
    x.append(float(input("Enter x: ")))
    y.append(float(input("Enter y: ")))

value = float(input("Enter interpolation value: "))

print("1. Newton Forward")
print("2. Newton Backward")
choice = int(input("Choose method (1 or 2): "))

h = x[1] - x[0]

# Create difference table
diff = [y.copy()]

for j in range(1, n):
    col = []
    for i in range(n - j):
        col.append(diff[j-1][i+1] - diff[j-1][i])
    diff.append(col)

# Forward interpolation
if choice == 1:
    u = (value - x[0]) / h
    result = y[0]
    term = 1

    for j in range(1, n):
        term *= (u - j + 1) / j
        result += term * diff[j][0]

# Backward interpolation
elif choice == 2:
    u = (value - x[n-1]) / h
    result = y[n-1]
    term = 1

    for j in range(1, n):
        term *= (u + j - 1) / j
        result += term * diff[j][n-j-1]

else:
    result = None
    print("Invalid choice")

if result is not None:
    print("Interpolated value =", result)

Enter number of data points:  5
Enter x:  0
Enter y:  1
Enter x:  1
Enter y:  2
Enter x:  2
Enter y:  5
Enter x:  3
Enter y:  10
Enter x:  4
Enter y:  17
Enter interpolation value:  0.5


1. Newton Forward
2. Newton Backward


Choose method (1 or 2):  1


Interpolated value = 1.25


In [ ]:
Newton Forward Interpolation

In [ ]:
n = int(input("Enter number of data points: "))

x = []
y = []

for i in range(n):
    x.append(float(input("Enter x: ")))
    y.append(float(input("Enter y: ")))

value = float(input("Enter interpolation value: "))

h = x[1] - x[0]
u = (value - x[0]) / h

diff = [y.copy()]

for j in range(1, n):
    col = []
    for i in range(n - j):
        col.append(diff[j-1][i+1] - diff[j-1][i])
    diff.append(col)

result = y[0]
term = 1

for j in range(1, n):
    term *= (u - j + 1) / j
    result += term * diff[j][0]

print("Interpolated value =", result)

Enter number of data points:  5
Enter x:  0
Enter y:  1
Enter x:  1
Enter y:  2
Enter x:  2
Enter y:  5
Enter x:  3
Enter y:  10
Enter x:  4
Enter y:  17
Enter interpolation value:  0.5


Interpolated value = 1.25


In [11]:
#Newton Forward Interpolation
# Function to convert floating number to integer format if whole number
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Inputs matching exact exam format
n = int(input("Enter number of points: "))
x = [float(v) for v in input("Enter x values: ").split()]
y = [float(v) for v in input("Enter y values: ").split()]
target_x_raw = parse_val(input("Enter x to find f(x): "))
target_x = float(target_x_raw)

# 2. 2D Forward Difference Table Calculation
diff = [[0.0] * n for _ in range(n)]

for i in range(n):
    diff[i][0] = y[i]

for j in range(1, n):
    for i in range(n - j):
        diff[i][j] = diff[i + 1][j - 1] - diff[i][j - 1]

# 3. Table Headers
headers = ["x", "Y"]
for j in range(1, n):
    if j == 1:
        headers.append("1st Diff")
    elif j == 2:
        headers.append("2nd Diff")
    elif j == 3:
        headers.append("3rd Diff")
    else:
        headers.append(f"{j}th Diff")

# Print Header
header_str = f"{headers[0]:<10}{headers[1]:<10}"
for h in headers[2:]:
    header_str += f"{h:<11}"

print(header_str.rstrip())
print("-" * len(header_str.rstrip()))

# Print Table Rows
for i in range(n):
    row_str = f"{x[i]:<10.4f}{y[i]:<10.4f}"
    for j in range(1, n - i):
        row_str += f"{diff[i][j]:<11.4f}"
    print(row_str.rstrip())

# 4. Newton's Forward Interpolation Calculation
h = x[1] - x[0]
u = (target_x - x[0]) / h

result = y[0]
u_term = 1.0

for j in range(1, n):
    u_term *= (u - j + 1) / j
    result += u_term * diff[0][j]

# 5. Output Result formatted to 4 decimal places
print()
print(f"f({target_x_raw}) = {result:.4f}")

Enter number of points: 5
Enter x values: 1891 1901 1911 1921 1931
Enter y values: 46 66 81 93 101
Enter x to find f(x): 1895
x         Y         1st Diff   2nd Diff   3rd Diff   4th Diff
-------------------------------------------------------------
1891.0000 46.0000   20.0000    -5.0000    2.0000     -3.0000
1901.0000 66.0000   15.0000    -3.0000    -1.0000
1911.0000 81.0000   12.0000    -4.0000
1921.0000 93.0000   8.0000
1931.0000 101.0000

f(1895) = 54.8528


In [7]:
# Newton Backward Interpolation

# Function to parse string number as int if whole, else float
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Inputs matching exact prompt format
n = int(input("Enter number of points: "))
x = [float(v) for v in input("Enter x values: ").split()]
y = [float(v) for v in input("Enter y values: ").split()]
target_x_raw = parse_val(input("Enter x to find f(x): "))
target_x = float(target_x_raw)

# 2. Construct 2D Backward Difference Table
diff = [[0.0] * n for _ in range(n)]

for i in range(n):
    diff[i][0] = y[i]

for j in range(1, n):
    for i in range(j, n):
        diff[i][j] = diff[i][j - 1] - diff[i - 1][j - 1]

# 3. Dynamic Column Headers
headers = ["x", "Y"]
for j in range(1, n):
    if j == 1:
        headers.append("1st Diff")
    elif j == 2:
        headers.append("2nd Diff")
    elif j == 3:
        headers.append("3rd Diff")
    else:
        headers.append(f"{j}th Diff")

# Print Table Header
header_str = f"{headers[0]:<10}{headers[1]:<10}"
for h in headers[2:]:
    header_str += f"{h:<11}"

print(header_str.rstrip())
print("-" * len(header_str.rstrip()))

# Print Row by Row
for i in range(n):
    row_str = f"{x[i]:<10.4f}{y[i]:<10.4f}"
    for j in range(1, i + 1):
        row_str += f"{diff[i][j]:<11.4f}"
    print(row_str.rstrip())

# 4. Calculate Newton's Backward Interpolation Formula
h = x[1] - x[0]
u = (target_x - x[n - 1]) / h

result = y[n - 1]
u_term = 1.0

for j in range(1, n):
    u_term *= (u + j - 1) / j
    result += u_term * diff[n - 1][j]

# 5. Output Result formatted to 4 decimal places
print()
print(f"f({target_x_raw}) = {result:.4f}")

Enter number of points: 5
Enter x values: 0 1 2 3 4
Enter y values: 1 3 11 31 69
Enter x to find f(x): 3.5
x         Y         1st Diff   2nd Diff   3rd Diff   4th Diff
-------------------------------------------------------------
0.0000    1.0000
1.0000    3.0000    2.0000
2.0000    11.0000   8.0000     6.0000
3.0000    31.0000   20.0000    12.0000    6.0000
4.0000    69.0000   38.0000    18.0000    6.0000     0.0000

f(3.5) = 47.3750


Bessel's Interpolation Formula

In [ ]:
# Bessel's Interpolation Formula

n = int(input("Enter number of data points (even number): "))

x = []
y = []

for i in range(n):
    x.append(float(input("Enter x: ")))
    y.append(float(input("Enter y: ")))

value = float(input("Enter value to interpolate: "))

h = x[1] - x[0]

# Find central index
k = n // 2 - 1

a = x[k]
u = (value - a) / h

# Create difference table
diff = [y.copy()]

for j in range(1, n):
    col = []
    for i in range(n - j):
        col.append(diff[j-1][i+1] - diff[j-1][i])
    diff.append(col)

# Bessel's formula
result = (y[k] + y[k+1]) / 2
result += (u - 0.5) * diff[1][k]

if n >= 3:
    result += (u * (u-1) / 2) * (
        diff[2][k-1] + diff[2][k]
    ) / 2

if n >= 4:
    result += ((u-0.5) * u * (u-1) / 6) * diff[3][k-1]

if n >= 5:
    result += ((u+1) * u * (u-1) * (u-2) / 24) * (
        diff[4][k-2] + diff[4][k-1]
    ) / 2

print("Interpolated value =", result)

Enter number of data points (even number):  6
Enter x:  0
Enter y:  1
Enter x:  1
Enter y:  2
Enter x:  2
Enter y:  5
Enter x:  3
Enter y:  10
Enter x:  4
Enter y:  17
Enter x:  5
Enter y:  26
Enter value to interpolate:  2.5


Interpolated value = 7.25


# Newton Divided Difference Interpolation

In [ ]:
n = int(input("Enter number of data points: "))

x = []
y = []

for i in range(n):
    x.append(float(input("Enter x: ")))
    y.append(float(input("Enter y: ")))

value = float(input("Enter value to interpolate: "))

# Create divided difference table
diff = [[0.0] * n for i in range(n)]

for i in range(n):
    diff[i][0] = y[i]

for j in range(1, n):
    for i in range(n - j):
        diff[i][j] = (
            diff[i+1][j-1] - diff[i][j-1]
        ) / (x[i+j] - x[i])

# Calculate interpolated value
result = diff[0][0]
term = 1

for j in range(1, n):
    term *= (value - x[j-1])
    result += term * diff[0][j]

print("Interpolated value =", result)

Enter number of data points:  5
Enter x:  5
Enter y:  150
Enter x:  7
Enter y:  392
Enter x:  11
Enter y:  1452
Enter x:  13
Enter y:  2366
Enter x:  21
Enter y:  9702
Enter value to interpolate:  6


Interpolated value = 252.0


In [12]:
# Helper to format numbers cleanly
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Dynamic Inputs matching exam standard
n = int(input("Enter number of points: "))
x = [float(v) for v in input("Enter x values: ").split()]
y = [float(v) for v in input("Enter y values: ").split()]
target_x_raw = parse_val(input("Enter x to find f(x): "))
target_x = float(target_x_raw)

# 2. Construct Divided Difference Table
diff = [[0.0] * n for _ in range(n)]

for i in range(n):
    diff[i][0] = y[i]

for j in range(1, n):
    for i in range(n - j):
        diff[i][j] = (diff[i + 1][j - 1] - diff[i][j - 1]) / (x[i + j] - x[i])

# 3. Dynamic Column Headers
headers = ["X", "f(x)"]
for j in range(1, n):
    if j == 1:
        headers.append("1st Diff")
    elif j == 2:
        headers.append("2nd Diff")
    elif j == 3:
        headers.append("3rd Diff")
    else:
        headers.append(f"{j}th Diff")

# Print Table Header
header_str = f"{headers[0]:<10}{headers[1]:<10}"
for h in headers[2:]:
    header_str += f"{h:<11}"

print(header_str.rstrip())
print("-" * len(header_str.rstrip()))

# Print Table Rows
for i in range(n):
    row_str = f"{x[i]:<10.4f}{y[i]:<10.4f}"
    for j in range(1, n - i):
        row_str += f"{diff[i][j]:<11.4f}"
    print(row_str.rstrip())

# 4. Calculate Interpolation using Divided Difference Formula
result = diff[0][0]
term = 1.0

for j in range(1, n):
    term *= (target_x - x[j - 1])
    result += term * diff[0][j]

# 5. Output Result formatted to 4 decimal places
print()
print(f"f({target_x_raw}) = {result:.4f}")

Enter number of points: 5
Enter x values: 5 7 11 13 21
Enter y values: 150 392 1452 2366 9702
Enter x to find f(x): 6
X         f(x)      1st Diff   2nd Diff   3rd Diff   4th Diff
-------------------------------------------------------------
5.0000    150.0000  121.0000   24.0000    1.0000     0.0000
7.0000    392.0000  265.0000   32.0000    1.0000
11.0000   1452.0000 457.0000   46.0000
13.0000   2366.0000 917.0000
21.0000   9702.0000

f(6) = 252.0000


Gauss Forward Interpolation


In [16]:
# Helper to format floating point numbers as int if whole
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# Factorial helper (avoiding external libraries)
def factorial(k):
    fact = 1
    for i in range(1, k + 1):
        fact *= i
    return fact

# 1. Inputs matching exact exam format
n = int(input("Enter number of points: "))
x = [float(v) for v in input("Enter x values: ").split()]
y = [float(v) for v in input("Enter y values: ").split()]
target_x_raw = parse_val(input("Enter x to find f(x): "))
target_x = float(target_x_raw)

# 2. Difference Table Calculation
diff = [[0.0] * n for _ in range(n)]

for i in range(n):
    diff[i][0] = y[i]

for j in range(1, n):
    for i in range(n - j):
        diff[i][j] = diff[i + 1][j - 1] - diff[i][j - 1]

# 3. Dynamic Column Headers
headers = ["x", "Y"]
for j in range(1, n):
    if j == 1:
        headers.append("1st Diff")
    elif j == 2:
        headers.append("2nd Diff")
    elif j == 3:
        headers.append("3rd Diff")
    else:
        headers.append(f"{j}th Diff")

# Print Header
header_str = f"{headers[0]:<10}{headers[1]:<10}"
for h in headers[2:]:
    header_str += f"{h:<11}"

print(header_str.rstrip())
print("-" * len(header_str.rstrip()))

# Print Difference Table Rows
for i in range(n):
    row_str = f"{x[i]:<10.4f}{y[i]:<10.4f}"
    for j in range(1, n - i):
        row_str += f"{diff[i][j]:<11.4f}"
    print(row_str.rstrip())

# 4. Find Origin x0 (closest point to target_x)
origin_idx = 0
min_diff = float('inf')
for i in range(n):
    diff_val = abs(x[i] - target_x)
    if diff_val < min_diff:
        min_diff = diff_val
        origin_idx = i

h = x[1] - x[0]
x0 = x[origin_idx]
u = (target_x - x0) / h

# 5. Gauss Forward Formula Calculation
# f(x) = y0 + u*Δy0 + u(u-1)/2! * Δ²y_(-1) + (u+1)u(u-1)/3! * Δ³y_(-1) + ...
result = y[origin_idx]
p = 1.0

for j in range(1, n):
    # Alternating factors: odd j adds (u + k), even j adds (u - k)
    if j % 2 == 1:
        p *= (u + (j - 1) // 2)
    else:
        p *= (u - j // 2)

    # Row index offset in difference table: row = origin_idx - (j // 2)
    row = origin_idx - (j // 2)

    if 0 <= row < n - j:
        term = (p / factorial(j)) * diff[row][j]
        result += term

# 6. Final Output
print()
print(f"f({target_x_raw}) = {result:.4f}")

Enter number of points: 5
Enter x values: 21 25 29 33 37
Enter y values: 18.4708 17.8144 17.1070 16.3432 15.5154
Enter x to find f(x): 30
x         Y         1st Diff   2nd Diff   3rd Diff   4th Diff
-------------------------------------------------------------
21.0000   18.4708   -0.6564    -0.0510    -0.0054    -0.0022
25.0000   17.8144   -0.7074    -0.0564    -0.0076
29.0000   17.1070   -0.7638    -0.0640
33.0000   16.3432   -0.8278
37.0000   15.5154

f(30) = 16.9216


Gauss Backward Interpolation

In [17]:
# Helper function to format whole floats as integers
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# Custom factorial helper function (no external library allowed)
def factorial(k):
    fact = 1
    for i in range(1, k + 1):
        fact *= i
    return fact

# 1. Input Section (Single-line space-separated format)
n = int(input("Enter number of points: "))
x = [float(v) for v in input("Enter x values: ").split()]
y = [float(v) for v in input("Enter y values: ").split()]
target_x_raw = parse_val(input("Enter x to find f(x): "))
target_x = float(target_x_raw)

# 2. Difference Table Calculation
diff = [[0.0] * n for _ in range(n)]

for i in range(n):
    diff[i][0] = y[i]

for j in range(1, n):
    for i in range(n - j):
        diff[i][j] = diff[i + 1][j - 1] - diff[i][j - 1]

# 3. Dynamic Column Headers & Table Printing
headers = ["x", "Y"]
for j in range(1, n):
    if j == 1:
        headers.append("1st Diff")
    elif j == 2:
        headers.append("2nd Diff")
    elif j == 3:
        headers.append("3rd Diff")
    else:
        headers.append(f"{j}th Diff")

header_str = f"{headers[0]:<10}{headers[1]:<10}"
for h in headers[2:]:
    header_str += f"{h:<11}"

print(header_str.rstrip())
print("-" * len(header_str.rstrip()))

for i in range(n):
    row_str = f"{x[i]:<10.4f}{y[i]:<10.4f}"
    for j in range(1, n - i):
        row_str += f"{diff[i][j]:<11.4f}"
    print(row_str.rstrip())

# 4. Find Origin x0 (Point closest to target_x)
origin_idx = 0
min_diff = float('inf')
for i in range(n):
    diff_val = abs(x[i] - target_x)
    if diff_val < min_diff:
        min_diff = diff_val
        origin_idx = i

h = x[1] - x[0]
x0 = x[origin_idx]
u = (target_x - x0) / h

# 5. Gauss Backward Formula Calculation
# f(x) = y0 + u*Δy_(-1) + u(u+1)/2! * Δ²y_(-1) + (u+1)u(u-1)/3! * Δ³y_(-2) + ...
result = y[origin_idx]
p = 1.0

for j in range(1, n):
    # Odd j: multiply by (u - (j-1)//2)
    # Even j: multiply by (u + j//2)
    if j % 2 == 1:
        p *= (u - (j - 1) // 2)
    else:
        p *= (u + j // 2)

    # Row index offset for Gauss Backward: origin_idx - (j + 1) // 2
    row = origin_idx - ((j + 1) // 2)

    if 0 <= row < n - j:
        term = (p / factorial(j)) * diff[row][j]
        result += term

# 6. Final Result Output
print()
print(f"f({target_x_raw}) = {result:.4f}")

Enter number of points: 6
Enter x values: 1931 1941 1951 1961 1971 1981
Enter y values: 12 15 20 27 39 52
Enter x to find f(x): 1966
x         Y         1st Diff   2nd Diff   3rd Diff   4th Diff   5th Diff
------------------------------------------------------------------------
1931.0000 12.0000   3.0000     2.0000     0.0000     3.0000     -10.0000
1941.0000 15.0000   5.0000     2.0000     3.0000     -7.0000
1951.0000 20.0000   7.0000     5.0000     -4.0000
1961.0000 27.0000   12.0000    1.0000
1971.0000 39.0000   13.0000
1981.0000 52.0000

f(1966) = 32.3438


Lagrange's Interpolation Formula

In [18]:
# Helper function to parse whole numbers as integers
def parse_val(s):
    val = float(s)
    return int(val) if val.is_integer() else val

# 1. Inputs matching exam standard
n = int(input("Enter number of points: "))
x = [float(v) for v in input("Enter x values: ").split()]
y = [float(v) for v in input("Enter y values: ").split()]
target_x_raw = parse_val(input("Enter x to find f(x): "))
target_x = float(target_x_raw)

# 2. Calculate Lagrange Interpolation
# f(x) = sum( y_i * product( (x - x_j) / (x_i - x_j) ) )
result = 0.0

for i in range(n):
    term = y[i]
    for j in range(n):
        if i != j:
            term *= (target_x - x[j]) / (x[i] - x[j])
    result += term

# 3. Final Output
print()
print(f"f({target_x_raw}) = {result:.4f}")

Enter number of points: 4
Enter x values: 5 6 9 11
Enter y values: 12 13 14 16
Enter x to find f(x): 10

f(10) = 14.6667
